# 🧠 How does a model see the world?

<a href="https://colab.research.google.com/github/racousin/scai_team/blob/main/how_models_see_the_world.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Every language model turns a text into a list of numbers — an **embedding**. That vector is, quite literally, how the model *sees* the text: two texts the model considers similar end up with nearby vectors.

In this notebook you will:

1. Take short bios — **the SCAI team**, cartoon characters, politicians.
2. Ask **two different small models** from 🤗 Hugging Face to read them.
3. Project each model's vectors down to 2D and **look at the world through each model's eyes**.
4. Then feed them **political opinions** and **conspiracy theories** — and catch each model's bias red-handed.

▶️ **Runtime ▸ Run all**, then scroll down to the plots (≈1 min on the free CPU runtime). After that, come back and start editing the texts.

In [10]:
!pip install -q sentence-transformers
!wget -q -O embedding_utils.py https://raw.githubusercontent.com/racousin/scai_team/main/embedding_utils.py
from embedding_utils import *

## 1. The texts

This is the only "data": one short bio per character, tagged with a `group`. Nothing else.

The team bios are built from the roles on [scai.sorbonne-universite.fr/management](https://scai.sorbonne-universite.fr/management) — on purpose, they say little more than a job title. Keep that in mind for later.

In [11]:
ITEMS = [
    # --- the SCAI team (roles from scai.sorbonne-universite.fr/management) ---
    {"name": "Gérard Biau", "group": "SCAI team",
     "text": "Director of SCAI, the Sorbonne Center for Artificial Intelligence in Paris."},
    {"name": "Xavier Fresquet", "group": "SCAI team",
     "text": "Deputy director of SCAI, the Sorbonne Center for Artificial Intelligence in Paris."},
    {"name": "Nora Roger", "group": "SCAI team",
     "text": "Administrative and financial manager of SCAI at Sorbonne University."},
    {"name": "Dimitris Alchatzidis", "group": "SCAI team",
     "text": "Project manager at SCAI, running artificial intelligence projects at Sorbonne University."},
    {"name": "Clotilde Chevet", "group": "SCAI team",
     "text": "Project manager at SCAI, coordinating interdisciplinary AI projects in Paris."},
    {"name": "Vivien Conti", "group": "SCAI team",
     "text": "Data scientist at SCAI, the Sorbonne Center for Artificial Intelligence."},
    {"name": "Omar El Dakkak", "group": "SCAI team",
     "text": "Associate professor in mathematics at Sorbonne University, member of the SCAI team."},
    {"name": "Baptiste Gregorutti", "group": "SCAI team",
     "text": "Research engineer at SCAI, the Sorbonne Center for Artificial Intelligence in Paris."},
    {"name": "Étienne Guével", "group": "SCAI team",
     "text": "Data scientist at SCAI, working on machine learning at Sorbonne University."},
    {"name": "Linghwei Loubaresse", "group": "SCAI team",
     "text": "Project manager at SCAI, the Sorbonne Center for Artificial Intelligence."},
    {"name": "Rakhee Patel", "group": "SCAI team",
     "text": "Strategic partnership manager at SCAI, connecting the center with its partners."},
    {"name": "Julie Perrin", "group": "SCAI team",
     "text": "Communication officer at SCAI, telling the story of AI research at Sorbonne University."},
    {"name": "Alain Rabaute", "group": "SCAI team",
     "text": "Manager of the SOUND.AI programme at SCAI, Sorbonne University."},
    {"name": "Julien Roudil", "group": "SCAI team",
     "text": "Operational manager of the Campus for AI professions and qualifications at Sorbonne University."},
    {"name": "Marco Salazar", "group": "SCAI team",
     "text": "Education project manager at SCAI, the Sorbonne Center for Artificial Intelligence."},
    # 👇 ADD YOURSELF — uncomment, put your name and a 1–2 sentence bio, re-run everything!
    # {"name": "YOUR NAME", "group": "SCAI team",
    #  "text": "I am ..."},

    # --- cartoons -----------------------------------------------------------
    {"name": "Homer Simpson", "group": "cartoon",
     "text": "Lazy but lovable father from Springfield who works at a nuclear power plant, loves donuts and beer, and always says d'oh."},
    {"name": "Mickey Mouse", "group": "cartoon",
     "text": "Cheerful cartoon mouse with big round ears and red shorts who whistles, laughs, and stars in Disney adventures."},
    {"name": "Naruto", "group": "cartoon",
     "text": "Hyperactive young ninja from the Hidden Leaf Village who eats ramen, never gives up, and dreams of becoming Hokage."},
    {"name": "Elsa", "group": "cartoon",
     "text": "Ice queen from Arendelle with magical powers to freeze everything around her, who sings about letting it go."},
    {"name": "SpongeBob", "group": "cartoon",
     "text": "Optimistic yellow sea sponge who lives in a pineapple under the sea and flips burgers at the Krusty Krab."},

    # --- politicians ---------------------------------------------------------
    {"name": "Macron (EN)", "group": "politician",
     "text": "President of France, former investment banker, who leads the country from the Élysée Palace in Paris."},
    {"name": "Macron (FR)", "group": "politician",
     "text": "Président de la République française, ancien banquier d'affaires, qui dirige le pays depuis le palais de l'Élysée à Paris."},
    {"name": "Barack Obama", "group": "politician",
     "text": "Former President of the United States, known for his eloquent speeches, healthcare reform, and a Nobel Peace Prize."},
    {"name": "Angela Merkel", "group": "politician",
     "text": "Former Chancellor of Germany, a trained physicist who led Europe's largest economy for sixteen years."},
    {"name": "Nelson Mandela", "group": "politician",
     "text": "South African anti-apartheid revolutionary who spent 27 years in prison and became his country's first Black president."},
]

## 2. The models

Two **small** models from the [Hugging Face hub](https://huggingface.co/models?library=sentence-transformers). Same job, different upbringing:

| | parameters | trained on |
|---|---|---|
| [`all-MiniLM-L6-v2`](https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2) | 22M | **English only** |
| [`paraphrase-multilingual-MiniLM-L12-v2`](https://huggingface.co/sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2) | 118M | **50+ languages** |

Any other model name from the hub works here too — swap one in whenever you like.

In [19]:
MODEL_A = "sentence-transformers/all-MiniLM-L6-v2"
MODEL_B = "cardiffnlp/twitter-roberta-base-sentiment-latest"

## 3. Two world views

Each model reads the same texts. Each dot below is one bio, placed on each model's own 2D map. **Hover a dot** to see the name and the full bio — in crowded spots only the outer names are printed, so if you can't find yours, it's hiding inside the team blob.

In [20]:
compare_models(ITEMS, MODEL_A, MODEL_B)

### What are you looking at?

- Each model turns a bio into a vector of 384 numbers; we squash those down to 2D. **The axes mean nothing — only distances do.** Close dots = "these texts look similar *to this model*".
- Do the three groups form clusters? In both models? Who ended up nearest to the SCAI team, and why?

## 4. Who is closest to whom?

The 2D map is a lossy summary. The matrix below is the real thing: the similarity the model assigns to **every pair** of texts, printed in each cell (1.00 = identical view, ~0 = unrelated). Swap in `MODEL_A` and see which numbers change the most.

In [22]:
similarity_heatmap(ITEMS, MODEL_A)

## 5. True or false? 🔬 vs 👁️

Below: 4 scientific statements and 4 conspiracy theories about the same topics. Will any model discover which is which?

In [23]:
CLAIMS = [
    {"name": "Moon — science 🔬", "group": "science",
     "text": "The Apollo 11 mission landed the first humans on the Moon in July 1969 and brought back 21.5 kg of lunar rock samples."},
    {"name": "Moon — conspiracy 👁️", "group": "conspiracy",
     "text": "Wake up: the Moon landing was filmed in a Hollywood studio, and NASA has been hiding the truth from us for decades."},
    {"name": "Earth — science 🔬", "group": "science",
     "text": "Satellite imagery and centuries of navigation confirm that the Earth is a sphere, slightly flattened at the poles."},
    {"name": "Earth — conspiracy 👁️", "group": "conspiracy",
     "text": "They don't want you to know it, but the Earth is flat and every government is paid to keep the lie alive."},
    {"name": "Vaccines — science 🔬", "group": "science",
     "text": "Large clinical trials on tens of thousands of volunteers showed that vaccines safely train the immune system to recognise a virus."},
    {"name": "Vaccines — conspiracy 👁️", "group": "conspiracy",
     "text": "Vaccines are a cover story: the elites use the injections to implant microchips and control the population."},
    {"name": "Climate — science 🔬", "group": "science",
     "text": "Global average temperature has risen by about 1.2 °C since the pre-industrial era, driven mainly by greenhouse gas emissions."},
    {"name": "Climate — conspiracy 👁️", "group": "conspiracy",
     "text": "Climate change is a hoax invented by scientists who get rich on research grants while they secretly control the weather."},
    # 🎓 The calm liar — uncomment, re-run, and watch where BOTH models file it:
    # {"name": "Moon — calm hoax 🎓", "group": "conspiracy",
    #  "text": "A careful review of the photographic evidence suggests that the lunar rock samples were most likely manufactured in laboratories on Earth."},
]

compare_models(CLAIMS, MODEL_A, MODEL_C)